In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "3c26be9efd017b4e918fba1f450f600ea03889b0"
assert len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH", "Pin the independently reviewed pushed implementation commit before Run all"
ACCOUNT_LABEL = ""  # A, B, or C
RUN_MODE = "fresh"  # fresh or resume
ATTEMPT_ID = ""  # required only for resume; copy the prior attempt folder name
STALE_MARKER_CONFIRMATION = ""  # CLEAR STALE MARKER only after the old runtime is stopped
assert ACCOUNT_LABEL in {"A", "B", "C"} and RUN_MODE in {"fresh", "resume"}
assert (RUN_MODE == "fresh" and not ATTEMPT_ID) or (RUN_MODE == "resume" and ATTEMPT_ID)
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
DIAGNOSTIC_VERSION = "v1_synthetic_mixture_dose_response"
SYNTHETIC_COUNTS = (0, 125, 250, 375, 500)
SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SHARED_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-coca-runs")
CLASSIFIER_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier"
V4_FAILURE_RECORD = CLASSIFIER_ROOT / "v4_focal_inverse_frequency" / "latest_validation_failure.json"
EMBEDDING_RECORD = CLASSIFIER_ROOT / "embedding_diagnostics" / "v1_frozen_coca_df_embeddings" / "latest_diagnostic_record.json"
DIAGNOSTIC_ROOT = CLASSIFIER_ROOT / "mixture_diagnostics" / DIAGNOSTIC_VERSION
LATEST_RECORD = DIAGNOSTIC_ROOT / "latest_diagnostic_record.json"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".coca_shared_root.json"
CANDIDATE_MANIFEST = SHARED_PROJECT_DIR / "outputs" / "exploratory_balanced_ddpm" / "sqrt_balanced_seed0_v1" / "candidate_synthetic_df" / "epoch0100_seed0" / "synthetic_df.csv"
EXPECTED_CANDIDATE_SHA256 = "9ef9b44e404f74aab8211f4e7d123da3258ba8ba4e3004a4147d1761ed343b34"


# CoCa v4 post-failure synthetic-mixture diagnostic

Exploratory dose-response only. Every condition keeps 585 train-df rows and changes only how many of the 500 extra slots use the fixed synthetic candidate. It uses validation only, never accesses test data, does not select a candidate, and cannot authorize formal training.

## Phase 0 CHECK — pinned code, prior evidence, and shared Drive

In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys
from datetime import datetime, timezone
from google.colab import drive, userdata
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project shortcut: {SHARED_PROJECT_DIR}"
assert SHARED_RUN_ROOT.is_dir(), f"missing shared run shortcut; do not create a private replacement: {SHARED_RUN_ROOT}"
assert SHARED_ROOT_SENTINEL.is_file() and V4_FAILURE_RECORD.is_file() and EMBEDDING_RECORD.is_file()
assert CANDIDATE_MANIFEST.is_file()
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
v4_failure = json.loads(V4_FAILURE_RECORD.read_text(encoding="utf-8"))
embedding = json.loads(EMBEDDING_RECORD.read_text(encoding="utf-8"))
assert v4_failure["validation_status"] == "VALIDATION FAILED" and v4_failure["formal_training_started"] is False
assert embedding["diagnostic_status"] == "COMPLETED" and embedding["test_data_accessed"] is False
assert embedding["analysis"]["group_counts"] == {"real_train_df": 85, "synthetic_df": 500, "validation_df": 14}
assert sha256(CANDIDATE_MANIFEST) == EXPECTED_CANDIDATE_SHA256
guard_paths = (V4_FAILURE_RECORD, EMBEDDING_RECORD)
guard_before = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
subprocess.run(["nvidia-smi"], check=True)
token = userdata.get("GH_TOKEN")
assert token and len(token) > 20, "Colab Secret GH_TOKEN with read access is required"
CODE_DIR = Path("/content/ddpm-coca-mixture-diagnostic-code")
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
credential = base64.b64encode(("x-access-token:" + token).encode()).decode()
clone_env = os.environ.copy(); clone_env["GIT_CONFIG_COUNT"] = "1"
clone_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
clone_env["GIT_CONFIG_VALUE_0"] = "Authorization: Basic " + credential
try:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True, env=clone_env)
finally:
    clone_env["GIT_CONFIG_VALUE_0"] = ""; token = credential = None; del token, credential, clone_env
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
remote = subprocess.check_output(["git", "-C", str(CODE_DIR), "remote", "get-url", "origin"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status and "@" not in remote and "x-access-token" not in remote
os.environ["HF_HOME"] = "/content/hf-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch==3.3.0", "pandas>=2.0", "pillow>=9.0"], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
from importlib.metadata import version
import pandas as pd, torch
from ddpm_derm import coca_run
assert torch.cuda.is_available() and version("open_clip_torch") == "3.3.0"
resolved_root = coca_run.require_existing_shared_root(SHARED_RUN_ROOT)
drive_probe = coca_run.probe_shared_drive(resolved_root)
sentinel = json.loads(SHARED_ROOT_SENTINEL.read_text(encoding="utf-8"))
assert sentinel["resolved_path"] == str(resolved_root)


## Phase 1 CHECK — copy only train/validation and candidate data locally

In [ ]:
import time
LOCAL_DATA_DIR = Path("/content/ham10000-train-val-only")
LOCAL_CANDIDATE_DIR = Path("/content/synthetic-candidate")
assert not LOCAL_DATA_DIR.exists() and not LOCAL_CANDIDATE_DIR.exists()
def copy_group(group, relatives, source_base, target_base, log_every=250):
    total = len(relatives)
    print(f"START {group} copy: total={total}", flush=True)
    start = time.perf_counter()
    for index, relative in enumerate(relatives, start=1):
        source = source_base / relative; target = target_base / relative
        try:
            if not source.is_file():
                raise FileNotFoundError(f"source image not found: {source}")
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
        except Exception as exc:
            raise RuntimeError(f"{group} copy failed at {index}/{total} for relative path {relative!r}: {exc}") from exc
        if index % log_every == 0 or index == total:
            elapsed = time.perf_counter() - start
            rate = index / elapsed if elapsed > 0 else float("inf")
            eta = (total - index) / rate if rate > 0 else float("inf")
            print(f"[{group}] {index}/{total} last={relative} elapsed={elapsed:.1f}s rate={rate:.1f} copies/s eta={eta:.1f}s", flush=True)
    elapsed = time.perf_counter() - start
    rate = total / elapsed if elapsed > 0 else float("inf")
    print(f"DONE {group} copy: total={total} elapsed={elapsed:.1f}s avg={rate:.1f} copies/s", flush=True)
(LOCAL_DATA_DIR / "manifests").mkdir(parents=True)
class_mapping_source = SHARED_PROJECT_DIR / "data" / "manifests" / "class_to_idx.json"
assert class_mapping_source.is_file(); shutil.copy2(class_mapping_source, LOCAL_DATA_DIR / "manifests" / "class_to_idx.json")
split_frames = {}
for split, expected_rows, expected_df in (("train", 6995, 85), ("val", 1510, 14)):
    source_manifest = SHARED_PROJECT_DIR / "data" / "manifests" / f"{split}.csv"
    frame = pd.read_csv(source_manifest)
    assert len(frame) == expected_rows and int((frame["dx"] == "df").sum()) == expected_df
    split_frames[split] = frame
    shutil.copy2(source_manifest, LOCAL_DATA_DIR / "manifests" / f"{split}.csv")
    copy_group(split, list(frame["image_path"].drop_duplicates()), SHARED_PROJECT_DIR / "data", LOCAL_DATA_DIR)
for field in ("image_id", "lesion_id"):
    assert not set(split_frames["train"][field]) & set(split_frames["val"][field])
candidate = pd.read_csv(CANDIDATE_MANIFEST)
assert len(candidate) == 500
for field in ("image_id", "lesion_id"):
    assert candidate[field].notna().all(), f"candidate {field} contains null values"
    assert candidate[field].astype(str).str.strip().ne("").all(), f"candidate {field} contains empty values"
assert not candidate["image_id"].duplicated().any(), "candidate image_id values must be unique"
for field in ("image_id", "lesion_id"):
    assert not set(candidate[field]) & set(split_frames["train"][field])
    assert not set(candidate[field]) & set(split_frames["val"][field])
LOCAL_CANDIDATE_MANIFEST = LOCAL_CANDIDATE_DIR / CANDIDATE_MANIFEST.name
LOCAL_CANDIDATE_DIR.mkdir(); shutil.copy2(CANDIDATE_MANIFEST, LOCAL_CANDIDATE_MANIFEST)
copy_group("candidate", list(candidate["image_path"]), CANDIDATE_MANIFEST.parent, LOCAL_CANDIDATE_DIR)
assert sha256(LOCAL_CANDIDATE_MANIFEST) == EXPECTED_CANDIDATE_SHA256
os.environ["DDPM_DERM_DATA_DIR"] = str(LOCAL_DATA_DIR)
from ddpm_derm import classifier_objective, classifier_run, manifests
fixed_split_identity = sha256(LOCAL_DATA_DIR / "manifests" / "train.csv")
mixture_specs = {}
for synthetic_count in SYNTHETIC_COUNTS:
    frame, spec = manifests.build_classifier_mixture_frame(df_target_count=585, synthetic_count=synthetic_count, seed=0, generated_manifest=LOCAL_CANDIDATE_MANIFEST)
    assert len(frame) == 7495 and manifests.class_counts(frame)["df"] == 585
    mixture_specs[synthetic_count] = spec
assert len({spec["candidate_order_sha256"] for spec in mixture_specs.values()}) == 1


## Phase 2 RUN CONTROL — isolated attempt, resume marker, and fixed queue

In [ ]:
assert not LATEST_RECORD.exists(), f"completed diagnostic already exists; do not overwrite: {LATEST_RECORD}"
if RUN_MODE == "fresh":
    prior = [] if not DIAGNOSTIC_ROOT.exists() else [path for path in DIAGNOSTIC_ROOT.iterdir() if path.is_dir()]
    assert not prior, f"prior incomplete attempts require explicit resume/review: {prior}"
    coca_run.ensure_tree(SHARED_RUN_ROOT, DIAGNOSTIC_ROOT.relative_to(SHARED_RUN_ROOT))
    ATTEMPT_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = DIAGNOSTIC_ROOT / ATTEMPT_ID
if RUN_MODE == "fresh": coca_run.ensure_tree(SHARED_RUN_ROOT, OUTPUT_DIR.relative_to(SHARED_RUN_ROOT))
else: assert OUTPUT_DIR.is_dir(), f"resume attempt not found: {OUTPUT_DIR}"
RUNNING_MARKER = OUTPUT_DIR / "_RUNNING.json"
IDENTITY_RECORD = OUTPUT_DIR / "diagnostic_identity.json"
COMPLETED_MARKER = OUTPUT_DIR / "_COMPLETED.json"
assert not COMPLETED_MARKER.exists()
diagnostic_identity = {"diagnostic_version": DIAGNOSTIC_VERSION, "git_commit": commit, "candidate_manifest_sha256": EXPECTED_CANDIDATE_SHA256, "fixed_split_identity": fixed_split_identity, "shared_root_uuid": sentinel["shared_root_uuid"], "synthetic_counts": list(SYNTHETIC_COUNTS), "seed": 0, "epochs": 5, "evaluation_scope": "validation_only", "interpretation_scope": "descriptive_not_candidate_selection"}
if RUN_MODE == "fresh": coca_run.write_json_atomic(IDENTITY_RECORD, diagnostic_identity)
else:
    assert IDENTITY_RECORD.is_file() and json.loads(IDENTITY_RECORD.read_text(encoding="utf-8")) == diagnostic_identity
    assert RUNNING_MARKER.is_file() and STALE_MARKER_CONFIRMATION == "CLEAR STALE MARKER", "stop the old runtime, then explicitly clear its stale marker"
    RUNNING_MARKER.unlink()
marker = {**coca_run.session_marker(ACCOUNT_LABEL, RUN_MODE, diagnostic_identity), "attempt_id": ATTEMPT_ID}
coca_run.create_running_marker(RUNNING_MARKER, marker)
print("fixed synthetic-count queue:", SYNTHETIC_COUNTS)
print("attempt id:", ATTEMPT_ID)
print("checkpoint cadence: every completed epoch; worst-case loss: one unfinished epoch")


## Phase 3 RUN — five validation-only frozen-CoCa conditions

In [ ]:
env = os.environ.copy(); env["PYTHONPATH"] = str(CODE_DIR / "src"); env["PYTHONUNBUFFERED"] = "1"
def paths_for(synthetic_count):
    label = f"S{synthetic_count:03d}"; root = OUTPUT_DIR / label
    checkpoints = root / "checkpoints" / "coca_vit_b32" / "C4_seed0"
    result = root / "results" / "coca_vit_b32" / "results_C4_seed0.json"
    return label, root, checkpoints / "best.pt", checkpoints / "last.pt", result
def validate_completed(synthetic_count):
    label, root, best, last, result_path = paths_for(synthetic_count)
    assert best.is_file() and last.is_file() and result_path.is_file()
    result = json.loads(result_path.read_text(encoding="utf-8")); identity = result["run_identity"]
    formal_output_identity = f"{sentinel['shared_root_uuid']}:sqrt_balanced_seed0_v1:coca_classifier:{DIAGNOSTIC_VERSION}:{ATTEMPT_ID}:{label}"
    assert identity["git_commit"] == commit and identity["run_version"] == DIAGNOSTIC_VERSION
    assert identity["candidate_manifest_sha256"] == EXPECTED_CANDIDATE_SHA256
    assert identity["fixed_split_identity"] == fixed_split_identity and identity["shared_root_uuid"] == sentinel["shared_root_uuid"]
    assert identity["formal_output_identity"] == formal_output_identity
    assert identity["data_intervention"] == mixture_specs[synthetic_count]
    assert identity["evaluation_scope"] == result["evaluation_scope"] == "validation_only"
    assert identity["training_objective"] == result["training_objective"]
    assert result["test_metrics"] is None and result["data_counts"] == {"train": 7495, "val": 1510, "test": None}
    assert len(result["history"]) == 5 and result["checkpoint_format"] == "frozen_backbone_head_only_v1"
    for path in (best, last):
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        assert checkpoint["run_identity"] == identity and checkpoint["checkpoint_format"] == "frozen_backbone_head_only_v1"
        assert "head_state_dict" in checkpoint and "model_state_dict" not in checkpoint and "encoder_state_dict" not in checkpoint
    assert torch.load(last, map_location="cpu", weights_only=False)["epoch"] == 5
    return result
for synthetic_count in SYNTHETIC_COUNTS:
    label, condition_root, best, last, result_path = paths_for(synthetic_count)
    if result_path.exists(): validate_completed(synthetic_count); print(f"[{label}] completed identity verified; skip"); continue
    if not condition_root.exists(): coca_run.ensure_tree(SHARED_RUN_ROOT, condition_root.relative_to(SHARED_RUN_ROOT))
    formal_output_identity = f"{sentinel['shared_root_uuid']}:sqrt_balanced_seed0_v1:coca_classifier:{DIAGNOSTIC_VERSION}:{ATTEMPT_ID}:{label}"
    command = [sys.executable, "-u", "-m", "ddpm_derm.train_classifier", "--arch", "coca_vit_b32", "--freeze-backbone", "--coca-pretrained", "laion2b_s13b_b90k", "--variant", "C4", "--generated-manifest", str(LOCAL_CANDIDATE_MANIFEST), "--mixture-synthetic-count", str(synthetic_count), "--seed", "0", "--epochs", "5", "--batch-size", "32", "--lr", "3e-4", "--weight-decay", "1e-4", "--df-target-count", "585", "--num-workers", "2", "--loss-name", "focal_cross_entropy", "--class-weighting", "inverse_frequency", "--focal-gamma", "2.0", "--evaluation-scope", "validation_only", "--output-dir", str(condition_root), "--run-label", f"coca_mixture_{label.lower()}", "--run-version", DIAGNOSTIC_VERSION, "--shared-root-uuid", sentinel["shared_root_uuid"], "--formal-output-identity", formal_output_identity, "--fixed-split-identity", fixed_split_identity, "--candidate-sha256", EXPECTED_CANDIDATE_SHA256, "--resume"]
    process = subprocess.Popen(command, cwd=CODE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout: print(line, end="", flush=True)
    if process.wait(): raise RuntimeError(f"mixture diagnostic failed: {label}")
    validate_completed(synthetic_count)


## Phase 4 REVIEW — descriptive sequence and immutable evidence boundary

In [ ]:
runs = {count: validate_completed(count) for count in SYNTHETIC_COUNTS}
objectives = [runs[count]["training_objective"] for count in SYNTHETIC_COUNTS]
assert all(objective == objectives[0] for objective in objectives[1:])
summary = []
for count in SYNTHETIC_COUNTS:
    result = runs[count]; matrix = result["validation_metrics"]["confusion_matrix"]
    predicted = [sum(row[index] for row in matrix) for index in range(len(matrix))]
    predicted_counts = {name: int(predicted[index]) for name, index in result["run_identity"]["class_mapping"].items()}
    summary.append({"synthetic_count": count, "duplicated_real_df_count": 500 - count, "best_validation_df_f1": result["best_val_df_f1"], "best_validation_macro_f1": result["validation_metrics"]["macro_f1"], "prediction_counts": predicted_counts, "non_collapse": bool(result["best_val_df_f1"] > 0 and predicted_counts["df"] > 0), "data_intervention": result["data_intervention"], "history": result["history"]})
df_f1_sequence = [row["best_validation_df_f1"] for row in summary]
record = {"diagnostic_status": "COMPLETED", **diagnostic_identity, "formal_training_started": False, "test_data_accessed": False, "interpretation_scope": "descriptive_not_candidate_selection", "conditions": summary, "df_f1_monotonic_nonincreasing_with_synthetic_count": all(left >= right for left, right in zip(df_f1_sequence, df_f1_sequence[1:])), "training_objective": objectives[0], "checkpoint_format": "frozen_backbone_head_only_v1", "drive_probes": drive_probe, "v4_failure_record": str(V4_FAILURE_RECORD), "embedding_diagnostic_record": str(EMBEDDING_RECORD), "diagnostic_artifact_directory": str(OUTPUT_DIR), "completed_utc": coca_run.utc_now()}
guard_after = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
assert guard_after == guard_before, "prior v4/embedding evidence changed"
coca_run.write_json_atomic(OUTPUT_DIR / "mixture_diagnostic.json", record)
coca_run.write_json_atomic(COMPLETED_MARKER, record)
coca_run.write_json_atomic(LATEST_RECORD, record)
RUNNING_MARKER.unlink()
print(json.dumps(summary, indent=2))
print("SYNTHETIC MIXTURE DIAGNOSTIC COMPLETED")
print("formal_training_started=false")
print("test_data_accessed=false")
print("No condition was selected and no formal run is authorized.")
print("Download this executed notebook plus mixture_diagnostic.json and the five condition result/checkpoint folders.")
